In [ ]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
csv_path = '/kaggle/input/q1-ka-ai-2026/Q1_data.csv'

df = pd.read_csv(csv_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Delivery_Time'].hist()

In [ ]:
df.head()

In [ ]:
df = df.drop("Order_ID", axis=1)


In [ ]:
df = df.dropna(subset=df.columns)


In [ ]:
df.info()

In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
      print("Dropping Duplicates...")
      df.drop_duplicates(inplace=True)
      print("Duplicates Dropped.")
    else:
      print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.info()

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")


In [ ]:
from sklearn.preprocessing import OneHotEncoder
OHE = OneHotEncoder(sparse_output=False)
catagorical = df.select_dtypes("object").columns
print(catagorical)
for col in catagorical:
  df[col] = OHE.fit_transform(df[[col]])

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]
X.head()

In [ ]:
folds = 5
kf = KFold(folds, shuffle=True)
model = RandomForestRegressor(n_estimators=200)
loss=[]
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{folds}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
  loss.append(mae)
print(loss)


In [ ]:
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
dt = pd.DataFrame({
    "predictions":y_pred,
    "truth":y_test}

)
dt.hist()

In [ ]:
%pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

cat = CatBoostRegressor(verbose=0)
rf = RandomForestRegressor(n_estimators=100)


In [ ]:
import numpy as np
folds = 5
kf = KFold(folds, shuffle=True)
model = RandomForestRegressor(n_estimators=200)
ensloss=[]
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{folds}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  cat.fit(X_train, y_train)
  rf.fit(X_train, y_train)
    # Predict
  catp = cat.predict(X_test)
  rfp = rf.predict(X_test)
  avg = (rfp + catp)/2

  # Calculate metrics
  mae = mean_absolute_error(y_test, avg)
  ensloss.append(mae)
print(ensloss)


In [ ]:
print(loss,'\n',ensloss)